### =============================================================================
###  SnapSquad — Face Detection Prototyping
###  Notebook: 01_face_detection.ipynb
###
###  Model: SCRFD-10G (high-accuracy, prototyping only)
###  Device model: SCRFD-2.5GF TFLite (3MB, ~40ms)
### =============================================================================

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import insightface
from insightface.app import FaceAnalysis
import os
import math

# ── Ensure CWD is the Notebooks folder (paths are relative) ─────────────────
os.chdir(r"C:\Users\kvpra\OneDrive\Desktop\snapsquad-app\Notebooks")

# Load SCRFD-10G
app = FaceAnalysis(allowed_modules=['detection'], providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

detector = insightface.model_zoo.get_model(
    r'models\scrfd_10g_bnkps.onnx',
    providers=['CPUExecutionProvider']
)
detector.prepare(ctx_id=0, input_size=(640, 640), det_thresh=0.5)
app.det_model = detector

# Load all images
test_folder = "test_images"
images = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]

# Grid layout — 3 images per row
cols = 3
rows = math.ceil(len(images) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes = np.array(axes).flatten()

for i, img_file in enumerate(images):
    img = cv2.imread(os.path.join(test_folder, img_file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces = app.get(img)
    img_draw = img_rgb.copy()

    for face in faces:
        box = face.bbox.astype(int)
        cv2.rectangle(img_draw, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 3)
        conf = round(face.det_score, 2)
        cv2.putText(img_draw, str(conf), (box[0], box[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        if face.kps is not None:
            for kp in face.kps.astype(int):
                cv2.circle(img_draw, tuple(kp), 5, (255, 255, 0), -1)

    axes[i].imshow(img_draw)
    axes[i].set_title(f"{img_file}  |  {len(faces)} face(s)", fontsize=11)
    axes[i].axis('off')

# Hide any unused grid slots
for j in range(len(images), len(axes)):
    axes[j].axis('off')

plt.suptitle("SCRFD-10G — Face Detection Grid (det_thresh=0.5)", fontsize=16)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'cv2'